# Event I/O Tutorial: Encoding, Formats & Throughput

Neural spike encoding, HDF5 event I/O, and throughput benchmarking.

In [ ]:
import numpy as np
from talon.io import encoding, h5, throughput

## 1. Neural Encoding

In [ ]:
data = np.random.rand(1, 28, 28).astype(np.float32)
spikes = encoding.rate_encode(data, n_steps=10)
print(f"Rate: {data.shape} -> {spikes.shape}, rate={spikes.mean():.3f}")

spk_lat = encoding.latency_encode(data, n_steps=10)
print(f"Latency: {spk_lat.shape}, rate={spk_lat.mean():.3f}")

frames = np.random.rand(10, 1, 28, 28).astype(np.float32)
ev = encoding.delta_encode(frames, threshold=0.1)
print(f"Delta: {frames.shape} -> {ev.shape}")

## 2. HDF5 I/O

In [ ]:
n = 5000
events = np.zeros(n, dtype=h5.EVENT_DTYPE)
events["x"] = np.random.randint(0, 28, n)
events["y"] = np.random.randint(0, 28, n)
events["t"] = np.sort(np.random.uniform(0, 1e6, n))
events["p"] = np.random.choice([-1, 1], n)

with h5.H5EventWriter("events.h5", sensor_size=(28, 28)) as w:
    w.write(events)
print(f"Wrote {n} events")

In [ ]:
reader = h5.H5EventReader("events.h5")
loaded = reader.read_all()
print(f"Read: {len(loaded)} events, fields={loaded.dtype.names}")

window = reader.read_time_window(100000, 500000)
print(f"Window [100k, 500k]: {len(window)} events")

## 3. Throughput Benchmark

In [ ]:
import dataclasses
result = throughput.benchmark_throughput(n_events=100000, sensor_size=(28, 28))
for f in dataclasses.fields(result):
    v = getattr(result, f.name)
    print(f"  {f.name}: {v:,.2f}" if isinstance(v, float) else f"  {f.name}: {v}")